# Assgn_013 — Reversible 20M LLM (Colab)

ERA V5 Session 13. Writes **versioned** folders under `reports/`:

```text
reports/v001_colab_t4_smoke/
reports/v002_colab_t4_full/
reports/LATEST.txt
```

PRIMARY PNG names stay stable inside each version folder.

**USD/hour (honest):** Colab T4 ≈ **0.12** USD/h (1.19 compute units × 0.10 USD/unit).
Free tier: set `ASSGN013_FREE_COLAB=1` (advertised 0.00; `usd_shadow` still uses 0.12).

```text
python tests/assgn013_rev_lab.py --smoke --tokens 4096
python tests/assgn013_rev_lab.py --full --tokens 50000000
```

Colab setup — upload Assign_0013 zip to /content first:

```text
# !unzip -q -o /content/Assign_0013_colab.zip -d /content/Assign_0013
# %cd /content/Assign_0013
# !cp -n .env.example .env
# !pip -q install -r requirements.txt
```

The zip ships **`.env.example` only** (never a real `.env`). Copy it on Colab, then set `HOST=colab` in the next cell.

Forbidden: relative paths like `content/Assign_0013` (creates `/content/content/...`).


In [ ]:
# Optional: persist reports to Drive so a disconnect does not lose PNGs
# from google.colab import drive
# drive.mount('/content/drive')
# REPORTS_ROOT = '/content/drive/MyDrive/Assign_0013/reports'
REPORTS_ROOT = None  # default: ROOT/reports

from pathlib import Path
import os
import shutil
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = Path.cwd().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Zip ships .env.example only — never a real .env
env_path = ROOT / '.env'
example = ROOT / '.env.example'
if not env_path.is_file() and example.is_file():
    shutil.copy2(example, env_path)
    print('created .env from .env.example')

os.environ.setdefault('ASSGN013_HOST', 'colab')
# T4 default; override after you see nvidia-smi / Colab banner
os.environ.setdefault('ASSGN013_USD_PER_HOUR', '0.12')
# Free Colab (no units): uncomment next line
# os.environ['ASSGN013_FREE_COLAB'] = '1'
# os.environ['ASSGN013_USD_PER_HOUR'] = '0.00'

# Keep file-based load_env_file in sync with Colab host/rate
if env_path.is_file():
    lines = []
    for line in env_path.read_text(encoding='utf-8').splitlines():
        if line.startswith('ASSGN013_HOST='):
            lines.append('ASSGN013_HOST=colab')
        elif line.startswith('ASSGN013_USD_PER_HOUR=') and os.environ.get('ASSGN013_USD_PER_HOUR'):
            lines.append(f"ASSGN013_USD_PER_HOUR={os.environ['ASSGN013_USD_PER_HOUR']}")
        else:
            lines.append(line)
    env_path.write_text('\n'.join(lines) + '\n', encoding='utf-8')

if REPORTS_ROOT:
    os.environ['ASSGN013_REPORTS_ROOT'] = REPORTS_ROOT

print('ROOT', ROOT, 'env_exists', env_path.is_file())


In [ ]:
# --- SMOKE batch (four runs, short tokens) ---
from IPython.display import Image, display
from src.pipeline.rev_lab import run_all, format_gate_print
from src.utils.versioning import read_latest

reports_root = Path(os.environ['ASSGN013_REPORTS_ROOT']) if os.environ.get('ASSGN013_REPORTS_ROOT') else (ROOT / 'reports')
results = run_all(reports_root, root=ROOT, tokens=4096, smoke=True)
print('\n'.join(format_gate_print(results)))
print('LATEST', read_latest(reports_root))


In [ ]:
# Display PRIMARY PNGs from the smoke version
vdir = Path(results['reports_dir'])
for name in (
    'assgn013_loss_trajectories.png',
    'assgn013_memory_vs_seq_batch.png',
    'assgn013_speed_cost_dashboard.png',
):
    display(Image(filename=str(vdir / name)))


## Full 50M-token batch (hours on T4)

Uncomment the next cell for the graded four-run protocol. To **resume** after a disconnect,
set `ASSGN013_RUN_VERSION` to the same `vNNN_colab_t4_full` id and re-run.


In [ ]:
# --- FULL batch (50M tokens x 4 runs) — uncomment to run ---
# RESUME = None  # e.g. 'v002_colab_t4_full'
# if RESUME:
#     os.environ['ASSGN013_RUN_VERSION'] = RESUME
# full = run_all(reports_root, root=ROOT, tokens=50_000_000, smoke=False, version=RESUME)
# print('\n'.join(format_gate_print(full)))
# print('LATEST', read_latest(reports_root))
# # zip for download
# import shutil
# shutil.make_archive(str(ROOT / full['version']), 'zip', full['reports_dir'])
# print('zipped', ROOT / (full['version'] + '.zip'))
